In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000013211A75930>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000013211A76020>, model_name='llama-3.1-8b-instant')

In [4]:
from langchain_core.messages import HumanMessage
model.invoke([
    HumanMessage(content="Hi, My name is Sejal and I am learning AI Engineering")
])

AIMessage(content="Hello Sejal, nice to meet you. AI Engineering is a fascinating field, and I'm happy to help you with any questions or topics you'd like to discuss. What specific areas of AI Engineering are you interested in learning about? Are you working on a project or have a particular goal in mind?", response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 48, 'total_tokens': 110, 'completion_time': 0.089719883, 'completion_tokens_details': None, 'prompt_time': 0.00500069, 'prompt_tokens_details': None, 'queue_time': 0.050364, 'total_time': 0.094720573}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': None}, id='run-d3276185-fb2e-4520-a0f8-5d5c7bd53c02-0', usage_metadata={'input_tokens': 48, 'output_tokens': 62, 'total_tokens': 110})

In [5]:
from langchain_core.messages import AIMessage
model.invoke([
    HumanMessage(content="Hi, My name is Sejal and I am learning AI Engineering"),
    AIMessage(content="Hello Sejal, nice to meet you. AI Engineering is a fascinating field that combines the principles of software engineering with the power of artificial intelligence. It's exciting that you're learning about it.\n\nWhat specifically are you looking to learn or achieve in AI Engineering? Are you focusing on a particular area such as natural language processing, computer vision, or reinforcement learning? Or are you looking to learn the fundamentals of AI and machine learning?\n\nI'm here to help and provide any guidance or resources you might need. What's on your mind?"),
    HumanMessage(content="Hey, what is my name and what I do?")
])

AIMessage(content='Your name is Sejal and you are currently learning AI Engineering.', response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 176, 'total_tokens': 190, 'completion_time': 0.029413141, 'completion_tokens_details': None, 'prompt_time': 0.211397656, 'prompt_tokens_details': None, 'queue_time': 0.050793951, 'total_time': 0.240810797}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': None}, id='run-d5b32702-665a-4fb2-9652-4fcb196c0e79-0', usage_metadata={'input_tokens': 176, 'output_tokens': 14, 'total_tokens': 190})

In [6]:
### Message History
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id: str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [7]:
config={"configurable":{"session_id":"chat1"}}

In [8]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Sejal and I am learning AI Engineering")],
    config=config
)

In [10]:
response.content

"Nice to meet you, Sejal. AI Engineering is a fascinating field that combines software engineering principles with artificial intelligence and machine learning. You must be excited to learn more about it. What specifically are you looking to learn in AI Engineering, or do you have any specific questions or topics you'd like to discuss?"

In [11]:
with_message_history.invoke(
    [HumanMessage(content="Whats my name?")],
    config=config
)

AIMessage(content='Your name is Sejal.', response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 144, 'total_tokens': 151, 'completion_time': 0.005284604, 'completion_tokens_details': None, 'prompt_time': 0.008691056, 'prompt_tokens_details': None, 'queue_time': 0.050446059, 'total_time': 0.01397566}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': None}, id='run-01ad8f69-1ea8-4d49-9667-06cb12affe8b-0', usage_metadata={'input_tokens': 144, 'output_tokens': 7, 'total_tokens': 151})

In [12]:
### change the config--->i.e change session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name?")],
    config=config1
)
response.content

"I don't have any information about your name as our conversation has just started. I'm a large language model, I don't have personal knowledge about individuals unless you choose to share it with me. If you'd like to share your name, I'd be happy to chat with you."

In [13]:
response=with_message_history.invoke(
    [HumanMessage(content="My name is Sejal")],
    config=config1
)
response.content

'Nice to meet you, Sejal. How are you doing today? Is there something I can help you with or would you like to chat?'

In [14]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name?")],
    config=config1
)
response.content

'Your name is Sejal.'

In [15]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are helpfull assistant.Answer all the questions to the best of your ability."),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt |model

In [17]:
chain.invoke({"messages": [HumanMessage(content="Hi My name is Sejal")]})

AIMessage(content="Nice to meet you, Sejal! I'm glad to be your helpful assistant. How can I assist you today?", response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 58, 'total_tokens': 83, 'completion_time': 0.050115456, 'completion_tokens_details': None, 'prompt_time': 0.151822017, 'prompt_tokens_details': None, 'queue_time': 0.244516849, 'total_time': 0.201937473}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': None}, id='run-877d310a-3db8-439b-9043-5b50fcd2690d-0', usage_metadata={'input_tokens': 58, 'output_tokens': 25, 'total_tokens': 83})

In [18]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [20]:
config={"configurable":{"session_id":"chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Sejal")],
    config=config
)
response.content

"We already introduced ourselves earlier, didn't we, Sejal? It's nice to see you again. How's your day going so far?"

In [21]:
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are helpfull assistant.Answer all the questions to the best of your ability in {languages}."),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt |model

In [22]:
response=chain.invoke(
    {"messages": [HumanMessage(content="Hi My name is Sejal")], 
     "languages": ["Marathi"]}
)
response.content

'नमस्कार! माझे नाव तुमच्या सेवेत असूनही, तुम्ही माझ्याशी किंवा कोणत्याही विषयावर बोलण्यास स्वागत आहे. तुम्ही काय विचारायचे आहे?'

In [23]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages")

In [24]:
config={"configurable":{"session_id":"chat4"}}
response=with_message_history.invoke(
    {"messages": [HumanMessage(content="Hi I am Sejal")], "languages": ["Marathi"]},
    config=config
)
response.content

'नमस्कार सेजल! तुमच्या मदतीसाठी कुठल्या विषयावर प्रश्न विचारत आहात का?'

In [25]:
response=with_message_history.invoke(
    {"messages": [HumanMessage(content="What is my name?")], "languages": ["Marathi"]},
    config=config
)
response.content

'तुझे नाव सेजल आहे.'

Managing Conversation History
1.trim_messages-->helper to reduce how many messages we are sending to the model.

In [ ]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=40,
    strategy="last",
    token_counter=model,
    include_system_messages=True,
    allow_partials=False,
    start_on="human"
)